# Resolving Duplicate Transactions

## 1. Business Problem

Amazon retail operations suspects duplicate transaction records in `amazon_customer_transactions.csv`.

**Task**
- Identify duplicate records
- Define a clear rule for which records to keep or remove, and why
- Calculate total revenue after cleaning
- Write a short closing note for the retail operations team
- Flag anything that should be double-checked before relying on the final number

**Columns**
- `transaction_id`
- `customer_id`
- `product`
- `amount`
- `order_timestamp`
- `recorded_at`

We will not treat `pandas.duplicated()` as the business answer. First we inspect the data, then we decide what “duplicate” means.

## 2. Dataset Overview

Load the CSV and look at the first rows. Goal in this step: confirm columns and see what a row looks like. No cleaning yet.

In [1]:
import pandas as pd
df = pd.read_csv("data/amazon_customer_transactions.csv")

In [2]:
df.head()

,transaction_id,customer_id,product,amount,order_timestamp,recorded_at
0,8113,311,Phone Case,21,2026-05-03 16:45,2026-05-03 16:48
1,8118,318,Webcam,79,2026-05-05 10:23,2026-05-05 10:25
2,8107,316,Yoga Mat,29,2026-05-02 20:33,2026-05-02 20:35
3,8137,302,Webcam,79,2026-05-16 23:36,2026-05-16 23:38
4,8133,304,Office Chair,230,2026-05-14 14:20,2026-05-14 14:22


## 3. Data Quality Checks

Check size, column types, and missing values before talking about duplicates.

In [3]:
print("Shape (rows, columns):", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())

Shape (rows, columns): (81, 6)

Data types:
transaction_id     int64
customer_id        int64
product              str
amount             int64
order_timestamp      str
recorded_at          str
dtype: object

Missing values per column:
transaction_id     0
customer_id        0
product            0
amount             0
order_timestamp    0
recorded_at        0
dtype: int64


### Exact duplicate rows

`duplicated()` here means every column matches. That is different from “same transaction recorded twice.”

In [4]:
exact_dup_count = df.duplicated().sum()
print("Exact duplicate rows (excluding first copy):", exact_dup_count)

df[df.duplicated(keep=False)].sort_values(list(df.columns))

Exact duplicate rows (excluding first copy): 3


,transaction_id,customer_id,product,amount,order_timestamp,recorded_at
7,8110,317,USB-C Cable,12,2026-05-03 03:42,2026-05-03 03:43
39,8110,317,USB-C Cable,12,2026-05-03 03:42,2026-05-03 03:43
15,8136,318,Backpack,55,2026-05-16 13:09,2026-05-16 13:12
28,8136,318,Backpack,55,2026-05-16 13:09,2026-05-16 13:12
34,8162,322,Webcam,79,2026-05-25 11:07,2026-05-25 11:08
51,8162,322,Webcam,79,2026-05-25 11:07,2026-05-25 11:08


### IDs

Check whether `transaction_id` is unique, and how often customers appear.

In [5]:
print("Rows:", len(df))
print("Unique transaction_id:", df["transaction_id"].nunique())
print("Unique customer_id:", df["customer_id"].nunique())

print("\ntransaction_id value counts (top):")
print(df["transaction_id"].value_counts().head(10))

print("\ncustomer_id value counts (top):")
print(df["customer_id"].value_counts().head(10))

Rows: 81
Unique transaction_id: 76
Unique customer_id: 23

transaction_id value counts (top):
transaction_id
8123    2
8110    2
8136    2
8162    2
8157    2
8113    1
8118    1
8107    1
8137    1
8133    1
Name: count, dtype: int64

customer_id value counts (top):
customer_id
322    6
301    6
318    5
317    5
306    5
305    5
312    5
316    4
302    4
304    4
Name: count, dtype: int64


### Repeated transaction_id groups

Same `transaction_id` more than once is a strong signal of a duplicate *record*, not just a customer buying twice.

In [6]:
dup_id_mask = df["transaction_id"].duplicated(keep=False)
df.loc[dup_id_mask].sort_values(["transaction_id", "recorded_at"])

,transaction_id,customer_id,product,amount,order_timestamp,recorded_at
7,8110,317,USB-C Cable,12,2026-05-03 03:42,2026-05-03 03:43
39,8110,317,USB-C Cable,12,2026-05-03 03:42,2026-05-03 03:43
64,8123,319,Phone Case,21,2026-05-09 10:12,2026-05-09 10:13
6,8123,319,Phone Case,30,2026-05-09 10:12,2026-05-09 16:13
15,8136,318,Backpack,55,2026-05-16 13:09,2026-05-16 13:12
28,8136,318,Backpack,55,2026-05-16 13:09,2026-05-16 13:12
78,8157,305,Desk Lamp,38,2026-05-23 14:38,2026-05-23 14:41
71,8157,305,Desk Lamp,33,2026-05-23 14:38,2026-05-23 20:41
34,8162,322,Webcam,79,2026-05-25 11:07,2026-05-25 11:08
51,8162,322,Webcam,79,2026-05-25 11:07,2026-05-25 11:08


### Parse timestamps

Convert the time columns so we can compare “which record arrived later.”

In [9]:
df["order_timestamp"] = pd.to_datetime(df["order_timestamp"])
df["recorded_at"] = pd.to_datetime(df["recorded_at"])
df[["order_timestamp", "recorded_at"]].dtypes

order_timestamp    datetime64[us]
recorded_at        datetime64[us]
dtype: object

In [ ]:
## 5. Duplicate Resolution Rule

Keep one row per `transaction_id`: the row with the latest `recorded_at`.

In [10]:
df_clean = (
    df.sort_values("recorded_at")
      .drop_duplicates(subset=["transaction_id"], keep="last")
      .sort_values("transaction_id")
      .reset_index(drop=True)
)

print("Rows before:", len(df))
print("Rows after:", len(df_clean))
print("Unique transaction_id after:", df_clean["transaction_id"].nunique())

Rows before: 81
Rows after: 76
Unique transaction_id after: 76


## 7. Revenue Before vs After Cleaning

`amount` summed on raw data can double-count exact copies and mix old plus corrected amounts.

In [12]:
revenue_before = df["amount"].sum()
revenue_after = df_clean["amount"].sum()

print("Revenue before cleaning:", revenue_before)
print("Revenue after cleaning:", revenue_after)
print("Difference (before - after):", revenue_before - revenue_after)

Revenue before cleaning: 4795
Revenue after cleaning: 4590
Difference (before - after): 205


## 8. Validation

Confirm the two IDs with different amounts kept the later recording.

In [13]:
check_ids = [8123, 8157]

print("Original pairs:")
display(df[df["transaction_id"].isin(check_ids)].sort_values(["transaction_id", "recorded_at"]))

print("Kept rows:")
display(df_clean[df_clean["transaction_id"].isin(check_ids)])

Original pairs:


,transaction_id,customer_id,product,amount,order_timestamp,recorded_at
64,8123,319,Phone Case,21,2026-05-09 10:12:00,2026-05-09 10:13:00
6,8123,319,Phone Case,30,2026-05-09 10:12:00,2026-05-09 16:13:00
78,8157,305,Desk Lamp,38,2026-05-23 14:38:00,2026-05-23 14:41:00
71,8157,305,Desk Lamp,33,2026-05-23 14:38:00,2026-05-23 20:41:00


Kept rows:


,transaction_id,customer_id,product,amount,order_timestamp,recorded_at
22,8123,319,Phone Case,30,2026-05-09 10:12:00,2026-05-09 16:13:00
56,8157,305,Desk Lamp,33,2026-05-23 14:38:00,2026-05-23 20:41:00


## 9. Key Findings

- Raw file: **81** rows. Cleaned file: **76** rows (one row per `transaction_id`).
- **3** transaction IDs were exact row copies (`8110`, `8136`, `8162`).
- **2** transaction IDs looked like corrections: same order time, later `recorded_at`, different `amount` (`8123`, `8157`).
- Revenue before cleaning: **4795**. After cleaning: **4590**. Difference: **205**.

## 10. Business Recommendation

Report **4590** as cleaned revenue, using latest `recorded_at` per `transaction_id`.

**Double-check before locking the number**
- Confirm with ops that a later `recorded_at` means a correction, not a second charge.
- Confirm `8123` (21 → 30) and `8157` (38 → 33) with source systems.
- Same customer appearing many times is not automatically a duplicate if `transaction_id` differs.
- Timestamps were stored as text; always parse them before choosing “latest.”